---
## 1. Introducción

**Objetivo:** aplicar PCA para reducir la dimensión de los datos; imágenes principalmente.

**PCA con scikit-learn:**

- El método directo (y eficiente) para calcular PCA se basa en hallar la SVD "completa" de la matriz de datos.
- La implementación de PCA en scikit-learn emplea un método aleatorizado, más eficiente que el directo.

**Aplicación a imágenes:**

- Usaremos datasets muy populares: MNIST, Fashion-MNIST y CIFAR-10.
- La dimensión de los datos en el caso de imágenes suele ser muy grande, lo que dificulta el aprendizaje de clasificadores.
- Una aproximación convencional para facilitar el aprendizaje consiste en reducir previamente la dimensión de los datos.
- Estudiaremos el error de reconstrucción en función de la dimensión reducida $K$.
- Interpretaremos las componentes principales como imágenes base (*eigenimages*).
- Reconstruiremos imágenes y las compararemos con sus originales.
- Estudiaremos la precisión de regresión logística en función de $K$.
- Aunque trabajemos con imágenes, se puede proceder de forma parecida con otro tipo de datos (de alta dimensión).

---
## 2. PCA con scikit-learn

### 2.1 Método directo

**Método directo para aplicar PCA:** a partir de la SVD "completa" de la matriz de datos.

**Ejemplo:** $N = 4$ datos de $D = 2$ dimensiones que queremos reducir a $K = 1$ dimensión.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([ [-1, -1], [0, 2], [2, 0], [3, 3] ])
N = len(X)

K = 1
_, _, Vt = np.linalg.svd(X - X.mean(0))
W = Vt[:K, :].T
Z = X @ W
hX = Z @ W.T
L = np.square(X - hX).sum(axis=1).mean()

fig, ax = plt.subplots(figsize=(3, 3))
ax.set_aspect("equal")
plt.axis([-1.5, 3.5, -1.5, 3.5])
plt.grid(True)
ax.set_title(f'$L(W)={L:.2f}$')
plt.scatter(*X.T, facecolor='white', edgecolor='k', s=100)
plt.scatter(*hX.T, facecolor='black', s=100, marker='x')
plt.show()

### 2.2 PCA de scikit-learn

**PCA con scikit-learn:** emplea un método aleatorizado para matrices grandes, más eficiente que el directo.

- **API:** <https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html>
- **Parámetros:** `n_components`, `svd_solver`, `random_state`, etc.
- **Atributos:** `components_`, `singular_values_`, `mean_`, `n_components`, etc.
- **Métodos:** `fit(X)`, `transform(X)`, `fit_transform(X)`, `inverse_transform(X)`, etc.

**Ejemplo (cont.):** con scikit-learn, `n_components` $= K$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X = np.array([ [-1, -1], [0, 2], [2, 0], [3, 3] ])
N = len(X)

K = 1
pca = PCA(n_components=K).fit(X - X.mean(0))
Z = pca.transform(X)
hX = pca.inverse_transform(Z)
L = np.square(X - hX).sum(axis=1).mean()

fig, ax = plt.subplots(figsize=(3, 3))
ax.set_aspect("equal")
plt.axis([-1.5, 3.5, -1.5, 3.5])
plt.grid(True)
ax.set_title(f'$L(W)={L:.2f}$')
plt.scatter(*X.T, facecolor='white', edgecolor='k', s=100)
plt.scatter(*hX.T, facecolor='black', s=100, marker='x')
plt.show()

---
## 3. MNIST

**Modified NIST (MNIST):** dataset de 70 000 imágenes $28 \times 28$ en gris de dígitos manuscritos.

- **Muy popular:** desde su introducción en los 90, MNIST ha sido muy usado para comparar técnicas de ML.
- **Partición estándar:** 60 000 primeras muestras para *training* y 10 000 restantes para *test*.
- **Agotado:** pues ya se han alcanzado tasas de error muy reducidas, por debajo del 0.1 %.

### 3.1 Carga del dataset

In [ ]:
import numpy as np
import datasets

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
ds

### 3.2 Primera muestra de entrenamiento

Salida esperada: `((28, 28), dtype('uint8'), np.int64(5))`

In [ ]:
train0 = ds['train'][0]
train0['image'].shape, train0['image'].dtype, train0['label']

### 3.3 Visualización de las primeras muestras

In [ ]:
import matplotlib.pyplot as plt

nrows = 10
ncols = 15
N = nrows * ncols

_, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 16*nrows/ncols), constrained_layout=True)
for ax, x, y in zip(axs.flat, ds['train'][:N]['image'], ds['train'][:N]['label']):
    ax.set_axis_off()
    ax.imshow(x, cmap=plt.cm.gray_r, interpolation="none")
    ax.set_title(y)

---
## 4. PCA aplicado a MNIST

### 4.1 Error de reconstrucción: estudio en función de $K$

Carga de los datos: cada imagen se aplana a un vector de $28 \cdot 28 = 784$ dimensiones y se normaliza a $[0, 1]$.

In [ ]:
import numpy as np
import datasets

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

Error de reconstrucción en *training* (azul, `+`) y *test* (magenta, `x`) para distintos valores de $K$.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

max_K = np.min(X_train.shape)
pca = PCA(n_components=max_K).fit(X_train)
Z_train = pca.transform(X_train)
Z_test = pca.transform(X_test)

Ks = np.array([100, 200, 300, 400, 500, 600, 700, max_K])
L_train = np.empty_like(Ks, dtype=float)
L_test = np.empty_like(Ks, dtype=float)
for i, K in enumerate(Ks):
    Z_train_K = Z_train.copy()
    Z_train_K[:, K:] = 0.0
    hX_train = pca.inverse_transform(Z_train_K)
    L_train[i] = np.square(X_train - hX_train).sum(axis=1).mean()

    Z_test_K = Z_test.copy()
    Z_test_K[:, K:] = 0.0
    hX_test = pca.inverse_transform(Z_test_K)
    L_test[i] = np.square(X_test - hX_test).sum(axis=1).mean()

plt.figure(figsize=(9, 2.5))
plt.grid(True)
plt.plot(Ks, L_train, '+-b', Ks, L_test, 'x:m')
plt.show()

### 4.2 Eigendigits y reconstrucción: con $K = 400$

Las diez primeras componentes principales representadas como imágenes (*eigendigits*).

In [ ]:
K = 400
pca = PCA(n_components=K).fit(X_train)
nrows, ncols = 1, 10

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 18*nrows/ncols), constrained_layout=True)
for i in range(min(K, nrows * ncols)):
    ax = axs.flat[i]
    ax.set_axis_off()
    ax.set_title(f"Eigendigit {i}")
    ax.imshow(pca.components_[i, :].reshape((28, 28)), cmap=plt.cm.gray, interpolation="none")

Las diez primeras imágenes de *test* (arriba) y su reconstrucción con $K = 400$ (abajo).

In [ ]:
ncols = 10
Z_test = pca.transform(X_test[:ncols])
hX_test = pca.inverse_transform(Z_test)

fig, axs = plt.subplots(nrows=2, ncols=10, figsize=(18, 18*2/ncols), constrained_layout=True)
for i in range(ncols):
    ax = axs.flat[i]
    ax.set_axis_off()
    ax.set_title(f"{y_test[i]} original")
    ax.imshow(X_test[i, :].reshape((28, 28)), cmap=plt.cm.gray, interpolation="none")

    ax = axs.flat[ncols + i]
    ax.set_axis_off()
    ax.set_title(f"{y_test[i]} reconstruido")
    ax.imshow(hX_test[i, :].reshape((28, 28)), cmap=plt.cm.gray, interpolation="none")

### 4.3 Precisión de regresión logística: estudio en función de $K$

Se entrena un clasificador de regresión logística sobre las $K$ primeras componentes principales.

In [ ]:
from sklearn.linear_model import LogisticRegression

max_K = np.min(X_train.shape)
pca = PCA(n_components=max_K).fit(X_train)
Z_train = pca.transform(X_train)
Z_test = pca.transform(X_test)

Ks = np.array([100, 200, 300, 400, 500, 600, 700, max_K])
acc_train = np.empty_like(Ks, dtype=float)
acc_test = np.empty_like(Ks, dtype=float)
for i, K in enumerate(Ks):
    clf = LogisticRegression(C=0.1, max_iter=200).fit(Z_train[:, :K], y_train)
    acc_train[i] = clf.score(Z_train[:, :K], y_train)
    acc_test[i] = clf.score(Z_test[:, :K], y_test)

plt.figure(figsize=(9, 2.5))
plt.grid(True)
plt.plot(Ks, 100. * acc_train, '+-b', Ks, 100. * acc_test, 'x:m')
plt.show()

Precisión en *test* para cada $K$. Salida esperada:

```text
100 92.1% 200 92.4% 300 92.5% 400 92.6% 500 92.6% 600 92.6% 700 92.6% 784 92.6%
```

In [ ]:
for i, K in enumerate(Ks):
    print(f'{K} {acc_test[i]:.1%}', end=' ')

---
## 5. Fashion-MNIST

**Fashion-MNIST:** MNIST con prendas de ropa; introducido en 2017 como sustituto del ya agotado MNIST.

### 5.1 Carga del dataset

In [ ]:
import numpy as np
import datasets

ds = datasets.load_dataset("zalando-datasets/fashion_mnist").with_format("numpy")
ds

### 5.2 Visualización de las primeras muestras

In [ ]:
import matplotlib.pyplot as plt

nrows = 4
ncols = 15
N = nrows * ncols
labels = ("T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot")

_, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 16*nrows/ncols), constrained_layout=True)
for ax, x, y in zip(axs.flat, ds['train'][:N]['image'], ds['train'][:N]['label']):
    ax.set_axis_off()
    ax.imshow(x, cmap=plt.cm.gray_r, interpolation="none")
    ax.set_title(labels[y])

---
## 6. CIFAR-10

**CIFAR-10:** dataset de 60 000 imágenes $32 \times 32$ a color de 10 clases; 6000 por clase.

### 6.1 Carga del dataset

In [ ]:
import numpy as np
import datasets

ds = datasets.load_dataset("uoft-cs/cifar10").with_format("numpy")
ds

### 6.2 Primera muestra de entrenamiento

Salida esperada: `((32, 32, 3), dtype('uint8'), np.int64(0))`

In [ ]:
train0 = ds['train'][0]
train0['img'].shape, train0['img'].dtype, train0['label']

### 6.3 Visualización de las primeras muestras

In [ ]:
import matplotlib.pyplot as plt

nrows = 9
ncols = 15
N = nrows * ncols
labels = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

_, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 16*nrows/ncols), constrained_layout=True)
for ax, x, y in zip(axs.flat, ds['train'][:N]['img'], ds['train'][:N]['label']):
    ax.set_axis_off()
    ax.imshow(x, cmap=plt.cm.gray_r, interpolation="none")
    ax.set_title(labels[y])

---
## 7. Ejercicio

**Ejercicio:** aplica PCA a Fashion-MNIST y CIFAR-10 siguiendo el ejemplo dado para MNIST.

**Aplicación a Fashion-MNIST:** en comparación con el ejemplo dado para MNIST

- Comprueba que, para obtener un error de reconstrucción prácticamente nulo, $K \geq 600$ (aprox.)
- Representa los diez primeros *eigenarticles* con $K = 600$
- Compara las diez primeras imágenes de test con sus reconstrucciones usando $K = 600$
- Comprueba que regresión logística supera ligeramente el 84 % de precisión en test con $K \geq 200$

**Aplicación a CIFAR-10:** en comparación con el ejemplo dado para MNIST

- Comprueba que, para obtener un error de reconstrucción prácticamente nulo, $K \geq 1000$ (aprox.)
- No trates de representar gráficamente los diez primeros *eigenobjects* (requiere un PCA por cada canal de color)
- Compara las diez primeras imágenes de test con sus reconstrucciones usando $K = 1000$
- Comprueba que regresión logística obtiene, aproximadamente, un 41 % de precisión en test con $K \geq 200$

### 7.1 Fashion-MNIST

#### 7.1.1 Carga de datos

Igual que en MNIST: imágenes $28 \times 28$ aplanadas a 784 dimensiones y normalizadas a $[0, 1]$.

In [ ]:
# Tu código aquí

#### 7.1.2 Error de reconstrucción: estudio en función de $K$

> Comprueba que, para obtener un error de reconstrucción prácticamente nulo, $K \geq 600$ (aprox.)

In [ ]:
# Tu código aquí

#### 7.1.3 Eigenarticles: con $K = 600$

> Representa los diez primeros *eigenarticles* con $K = 600$

In [ ]:
# Tu código aquí

#### 7.1.4 Reconstrucción: con $K = 600$

> Compara las diez primeras imágenes de test con sus reconstrucciones usando $K = 600$

In [ ]:
# Tu código aquí

#### 7.1.5 Precisión de regresión logística: estudio en función de $K$

> Comprueba que regresión logística supera ligeramente el 84 % de precisión en test con $K \geq 200$

Salida esperada:

```text
Precisión en test con K = 200: 84.34%
```

In [ ]:
# Tu código aquí

### 7.2 CIFAR-10

#### 7.2.1 Carga de datos

Imágenes $32 \times 32$ a color: se aplanan a $32 \cdot 32 \cdot 3 = 3072$ dimensiones y se normalizan a $[0, 1]$.

In [ ]:
# Tu código aquí

#### 7.2.2 Error de reconstrucción: estudio en función de $K$

> Comprueba que, para obtener un error de reconstrucción prácticamente nulo, $K \geq 1000$ (aprox.)

In [ ]:
# Tu código aquí

#### 7.2.3 Reconstrucción: con $K = 1000$

> No trates de representar gráficamente los diez primeros *eigenobjects* (requiere un PCA por cada canal de color).
>
> Compara las diez primeras imágenes de test con sus reconstrucciones usando $K = 1000$

In [ ]:
# Tu código aquí

#### 7.2.4 Precisión de regresión logística: estudio en función de $K$

> Comprueba que regresión logística obtiene, aproximadamente, un 41 % de precisión en test con $K \geq 200$

In [ ]:
# Tu código aquí